# 01 - Paramétrage et lancement du contrôle RGPD Article 9

Notebook de pilotage local : référentiels, scoring, lemmatisation spaCy, familles lexicales et lancement du batch.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import yaml

PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
sys.path.append(str(SRC_DIR))

PDF_DIR = PROJECT_ROOT / "input" / "pdf"
REFERENTIEL_DIR = PROJECT_ROOT / "input" / "referentiel"
CONFIG_FILE = PROJECT_ROOT / "config" / "scoring.yaml"
OUTPUT_CSV = PROJECT_ROOT / "output" / "resultats_controle.csv"
OUTPUT_XLSX = PROJECT_ROOT / "output" / "resultats_controle.xlsx"

print("Projet :", PROJECT_ROOT)

## 1. Vérification des dépendances locales

In [ ]:
# Si spaCy ou le modèle français ne sont pas installés, exécuter dans un terminal :
# pip install -r ../requirements.txt
# python -m spacy download fr_core_news_md

import spacy
try:
    nlp = spacy.load("fr_core_news_md", disable=["ner"])
    print("Modèle spaCy français OK : fr_core_news_md")
except OSError as exc:
    raise RuntimeError("Modèle spaCy manquant. Exécuter : python -m spacy download fr_core_news_md") from exc

## 2. Vérifier les PDF et référentiels

In [ ]:
pdf_files = sorted(PDF_DIR.glob("*.pdf"))
print(f"Nombre de PDF trouvés : {len(pdf_files)}")
for pdf in pdf_files[:20]:
    print("-", pdf.name)

required_files = [
    REFERENTIEL_DIR / "mots_interdits.csv",
    REFERENTIEL_DIR / "synonymes.csv",
    REFERENTIEL_DIR / "familles_lexicales.csv",
    REFERENTIEL_DIR / "whitelist.csv",
    CONFIG_FILE,
]

for file in required_files:
    print(file.name, "OK" if file.exists() else "MANQUANT")

## 3. Charger les référentiels métier

In [ ]:
mots_interdits = pd.read_csv(REFERENTIEL_DIR / "mots_interdits.csv", encoding="utf-8")
synonymes = pd.read_csv(REFERENTIEL_DIR / "synonymes.csv", encoding="utf-8")
familles_lexicales = pd.read_csv(REFERENTIEL_DIR / "familles_lexicales.csv", encoding="utf-8")
whitelist = pd.read_csv(REFERENTIEL_DIR / "whitelist.csv", encoding="utf-8")

display(mots_interdits.head(20))
display(synonymes.head(20))
display(familles_lexicales.head(20))
display(whitelist.head(20))

## 4. Paramétrer le scoring

In [ ]:
with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# Paramètres modifiables pour ce lancement
config["detection"]["activer_lemmatisation"] = True
config["detection"]["activer_familles_lexicales"] = True
config["detection"]["activer_synonymes"] = True
config["detection"]["activer_mots_proches"] = True
config["detection"]["seuil_similarite_orthographique"] = 88
config["detection"]["taille_contexte_mots"] = 12

config

## 5. Test rapide de lemmatisation

In [ ]:
from lemmatizer import lemmatize_text

texte_test = "Le client indique être malade et suivre un traitement."
tokens = lemmatize_text(texte_test)
pd.DataFrame(tokens)

## 6. Lancer le traitement

In [ ]:
from extract_pdf import extract_text_from_pdf
from load_referential import load_referentials
from detection import analyze_document
from export_results import export_results

referentials = load_referentials(REFERENTIEL_DIR)
results = []

for pdf_path in sorted(PDF_DIR.glob("*.pdf")):
    print("Analyse :", pdf_path.name)
    text_by_page = extract_text_from_pdf(pdf_path)
    results.extend(analyze_document(pdf_path, text_by_page, referentials, config))

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
export_results(results, OUTPUT_CSV, OUTPUT_XLSX)
print("CSV généré :", OUTPUT_CSV)
print("Excel généré :", OUTPUT_XLSX)

## 7. Aperçu des résultats

In [ ]:
df_results = pd.read_csv(OUTPUT_CSV, encoding="utf-8-sig")
display(df_results.head(50))
print("Documents :", df_results["document"].nunique())
print("Alertes :", len(df_results[df_results["score"] > 0]))